In [0]:
%pip install sentence-transformers

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    ArrayType,
    FloatType
)

from sentence_transformers import SentenceTransformer


SILVER_TABLE = "career_os.silver.jobs_clean"
EMBEDDINGS_TABLE = "career_os.ai.job_embeddings"
MODEL_NAME = "all-MiniLM-L6-v2"


jobs_df = (
    spark.table(SILVER_TABLE)
    .select("job_id","search_text")
    .where("job_id IS NOT NULL AND search_text IS NOT NULL")
)
model = SentenceTransformer(MODEL_NAME)
jobs = jobs_df.collect()
embeddings = []
for row in jobs:
    vector = model.encode(row.search_text).tolist()
    embeddings.append((row.job_id, row.search_text, vector))

print(f"Generated embeddings for " f"{len(embeddings)} jobs")

embedding_schema = StructType([
    StructField("job_id",StringType(),False),
    StructField("search_text",StringType(),True),
    StructField("embedding",ArrayType(FloatType()),False)
])

embedding_df = spark.createDataFrame(embeddings,schema=embedding_schema)


(
    embedding_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true"
    )
    .saveAsTable(EMBEDDINGS_TABLE)
)